# 04 — Business questions (Procurement)

Run after `01` → `02` → `03` with the same `target_catalog` / `target_schema`.

**Definitions:**
- **Spend** = `l_quantity * ps_supplycost` for the line item's (part, supplier) pair.
- **Inventory value** = `ps_availqty * ps_supplycost`.
- **Supplier with complaints** = `s_comment` contains `Customer ... Complaints`.
- Trends over time use **complete quarters** only.

In [0]:
import re

import matplotlib.pyplot as plt
import pandas as pd

_user = spark.sql("SELECT current_user()").first()[0]
_default_schema = "lakehouse_" + re.sub(r"\W+", "_", _user.split("@")[0]).lower()

dbutils.widgets.text("target_catalog", "workspace", "Target catalog")
dbutils.widgets.text("target_schema", _default_schema, "Target schema")

TARGET_CATALOG = dbutils.widgets.get("target_catalog")
TARGET_SCHEMA = dbutils.widgets.get("target_schema")


def bronze(table: str) -> str:
    return f"{TARGET_CATALOG}.{TARGET_SCHEMA}.bronze_{table}"


def silver(table: str) -> str:
    return f"{TARGET_CATALOG}.{TARGET_SCHEMA}.silver_{table}"


def gold(table: str) -> str:
    return f"{TARGET_CATALOG}.{TARGET_SCHEMA}.gold_{table}"


BRAND = "Brand#32"
TOP_N = 10
TOLERANCE = 0.01

## Profiling

### Suppliers per part

In [0]:
display(spark.sql(f"""
    SELECT layer, suppliers_per_part, COUNT(*) AS parts
    FROM (
        SELECT 'bronze' AS layer, ps_partkey, COUNT(*) AS suppliers_per_part
        FROM {bronze('partsupp')} GROUP BY ps_partkey
        UNION ALL
        SELECT 'silver', ps_partkey, COUNT(*)
        FROM {silver('partsupp')} GROUP BY ps_partkey
    )
    GROUP BY layer, suppliers_per_part
    ORDER BY layer, suppliers_per_part
"""))

In bronze every part has 4 suppliers. After quarantine, 10 parts have only 1 valid supplier left and 239 have 2.

### Supply cost vs retail price

In [0]:
display(spark.sql(f"""
    SELECT COUNT(*) AS offers,
           MIN(ps.ps_supplycost) AS min_supply_cost,
           MAX(ps.ps_supplycost) AS max_supply_cost,
           MIN(p.p_retailprice) AS min_retail_price,
           MAX(p.p_retailprice) AS max_retail_price,
           COUNT_IF(ps.ps_supplycost <= 0) AS non_positive_cost,
           COUNT_IF(ps.ps_supplycost > p.p_retailprice) AS cost_above_retail,
           ROUND(COUNT_IF(ps.ps_supplycost > p.p_retailprice) / COUNT(*) * 100, 4) AS cost_above_retail_pct
    FROM {bronze('partsupp')} ps
    JOIN {bronze('part')} p ON p.p_partkey = ps.ps_partkey
"""))

3,479 offers (**0.087%**) have supply cost above retail price. They are moved to quarantine in silver. No cost is ≤ 0.

### Complaints in free text

In [0]:
display(spark.sql(f"""
    SELECT COUNT(*) AS suppliers,
           COUNT_IF(s_comment LIKE '%Customer%Complaints%') AS customer_complaints,
           COUNT_IF(LOWER(s_comment) LIKE '%complain%') AS any_complain_word,
           COUNT_IF(s_comment LIKE '%Customer%Recommends%') AS customer_recommends
    FROM {bronze('supplier')}
"""))

display(spark.sql(f"""
    SELECT s_suppkey, s_comment
    FROM {bronze('supplier')}
    WHERE LOWER(s_comment) LIKE '%complain%'
    ORDER BY s_suppkey
    LIMIT 10
"""))

The pattern `Customer ... Complaints` finds **26** suppliers. A broader case-insensitive search finds the same 26, so the pattern is complete. 26 more suppliers have `Customer ... Recommends` (positive).

### Partial periods

In [0]:
spark.sql(f"""
    SELECT grain, period_start, calendar_days, days_with_data,
           days_with_data = calendar_days AS is_complete
    FROM (
        SELECT 'month' AS grain,
               CAST(DATE_TRUNC('month', o_orderdate) AS DATE) AS period_start,
               DATEDIFF(ADD_MONTHS(CAST(DATE_TRUNC('month', o_orderdate) AS DATE), 1),
                        CAST(DATE_TRUNC('month', o_orderdate) AS DATE)) AS calendar_days,
               COUNT(DISTINCT o_orderdate) AS days_with_data
        FROM {silver('orders')}
        GROUP BY 2, 3
        UNION ALL
        SELECT 'quarter',
               CAST(DATE_TRUNC('quarter', o_orderdate) AS DATE),
               DATEDIFF(ADD_MONTHS(CAST(DATE_TRUNC('quarter', o_orderdate) AS DATE), 3),
                        CAST(DATE_TRUNC('quarter', o_orderdate) AS DATE)),
               COUNT(DISTINCT o_orderdate)
        FROM {silver('orders')}
        GROUP BY 2, 3
    )
""").createOrReplaceTempView("period_completeness")

spark.sql("""
    SELECT period_start AS quarter_start
    FROM period_completeness
    WHERE grain = 'quarter' AND is_complete
""").createOrReplaceTempView("complete_quarters")

display(spark.sql("""
    SELECT grain, period_start, calendar_days, days_with_data
    FROM period_completeness
    WHERE NOT is_complete
    ORDER BY grain, period_start
"""))

Only 1998-08 and 1998-Q3 are partial. They are excluded from trends.

## Reconciliation across layers

Check: bronze = silver + quarantine, silver = gold.

In [0]:
def scalar(query: str):
    value = spark.sql(query).first()[0]
    return value if value is not None else 0


bronze_unmatched_lines = scalar(f"""
    SELECT COUNT(*) FROM {bronze('lineitem')} l
    LEFT ANTI JOIN {bronze('partsupp')} ps
      ON ps.ps_partkey = l.l_partkey AND ps.ps_suppkey = l.l_suppkey
""")

spend = {
    "bronze": scalar(f"""
        SELECT SUM(l.l_quantity * ps.ps_supplycost)
        FROM {bronze('lineitem')} l
        JOIN {bronze('partsupp')} ps
          ON ps.ps_partkey = l.l_partkey AND ps.ps_suppkey = l.l_suppkey"""),
    "silver": scalar(f"""
        SELECT SUM(l.l_quantity * ps.ps_supplycost)
        FROM {silver('lineitem')} l
        JOIN {silver('partsupp')} ps
          ON ps.ps_partkey = l.l_partkey AND ps.ps_suppkey = l.l_suppkey"""),
    "quarantine": scalar(f"""
        SELECT SUM(l.l_quantity * q.ps_supplycost)
        FROM {silver('lineitem_quarantine')} l
        JOIN {silver('partsupp_quarantine')} q
          ON q.ps_partkey = l.l_partkey AND q.ps_suppkey = l.l_suppkey"""),
    "gold": scalar(f"SELECT SUM(spend) FROM {gold('part_supplier')}"),
}

inventory = {
    "bronze": scalar(f"SELECT SUM(ps_availqty * ps_supplycost) FROM {bronze('partsupp')}"),
    "silver": scalar(f"SELECT SUM(ps_availqty * ps_supplycost) FROM {silver('partsupp')}"),
    "quarantine": scalar(f"SELECT SUM(ps_availqty * ps_supplycost) FROM {silver('partsupp_quarantine')}"),
    "gold": scalar(f"SELECT SUM(inventory_value) FROM {gold('supplier')}"),
}

reconciliation = []
for metric, v in [("spend", spend), ("inventory value", inventory)]:
    reconciliation.append((
        metric,
        float(v["bronze"]), float(v["silver"]), float(v["quarantine"]), float(v["gold"]),
        round(float(v["quarantine"]) / float(v["bronze"]) * 100, 4),
        abs(v["bronze"] - v["silver"] - v["quarantine"]) <= TOLERANCE,
        abs(v["silver"] - v["gold"]) <= TOLERANCE,
    ))

display(spark.createDataFrame(
    reconciliation,
    "metric STRING, bronze DOUBLE, silver DOUBLE, quarantine DOUBLE, gold DOUBLE, "
    "quarantine_share_pct DOUBLE, bronze_eq_silver_plus_quarantine BOOLEAN, silver_eq_gold BOOLEAN",
))

assert bronze_unmatched_lines == 0, f"{bronze_unmatched_lines} bronze line items have no partsupp pair"
assert all(r[6] and r[7] for r in reconciliation), "Layers do not reconcile"

Both metrics reconcile. Quarantine holds **0.17%** of spend and inventory value; it is excluded from gold.

## Q1. Top-10 suppliers by inventory value

In [0]:
q1 = spark.sql(f"""
    WITH ranked AS (
        SELECT supplier_id, supplier_name, nation, region, inventory_value,
               inventory_value / SUM(inventory_value) OVER () * 100 AS share_pct,
               ROW_NUMBER() OVER (ORDER BY inventory_value DESC) AS rank
        FROM {gold('supplier')}
    )
    SELECT rank, supplier_id, supplier_name, nation, region, inventory_value, ROUND(share_pct, 4) AS share_pct
    FROM ranked
    WHERE rank <= {TOP_N}
    ORDER BY rank
""")
display(q1)

display(spark.sql(f"""
    WITH ranked AS (
        SELECT inventory_value, ROW_NUMBER() OVER (ORDER BY inventory_value DESC) AS rank
        FROM {gold('supplier')}
    )
    SELECT COUNT(*) AS suppliers,
           SUM(inventory_value) AS total_inventory_value,
           SUM(CASE WHEN rank <= {TOP_N} THEN inventory_value END) AS top_inventory_value,
           ROUND(SUM(CASE WHEN rank <= {TOP_N} THEN inventory_value END) / SUM(inventory_value) * 100, 4) AS top_share_pct,
           ROUND({TOP_N} / COUNT(*) * 100, 4) AS equal_share_pct
    FROM ranked
"""))

In [0]:
q1_pd = q1.toPandas().iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(q1_pd["supplier_name"], q1_pd["inventory_value"].astype(float) / 1e6)
ax.set_title(f"Top-{TOP_N} suppliers by inventory value (available quantity × supply cost)")
ax.set_xlabel("Inventory value, millions")
plt.tight_layout()
plt.show()

**Answer:** the top-10 hold **2.76B of 10.00T (0.028%)**, vs 0.02% if all suppliers were equal. Inventory is spread evenly; there is no dependency on a few suppliers. 6 of the top 10 are in Africa.

## Q2. Brand#32: supply cost spread and the cheapest supplier

In [0]:
spark.sql(f"""
    SELECT part_id, part_name,
           MAX(suppliers_offering) AS suppliers_offering,
           MIN(min_supply_cost) AS min_supply_cost,
           MAX(max_supply_cost) AS max_supply_cost,
           MAX(max_supply_cost) - MIN(min_supply_cost) AS cost_spread,
           (MAX(max_supply_cost) - MIN(min_supply_cost)) / MIN(min_supply_cost) * 100 AS cost_spread_pct
    FROM {gold('part_supplier')}
    WHERE brand = '{BRAND}'
    GROUP BY part_id, part_name
""").createOrReplaceTempView("brand_part_costs")

display(spark.sql("""
    SELECT COUNT(*) AS parts,
           ROUND(AVG(suppliers_offering), 2) AS avg_suppliers_per_part,
           ROUND(AVG(cost_spread), 2) AS avg_cost_spread,
           ROUND(AVG(cost_spread_pct), 2) AS avg_cost_spread_pct,
           ROUND(PERCENTILE(CAST(cost_spread_pct AS DOUBLE), 0.5), 2) AS median_cost_spread_pct,
           ROUND(MAX(max_supply_cost / min_supply_cost), 2) AS max_cost_ratio
    FROM brand_part_costs
    WHERE suppliers_offering > 1
"""))

display(spark.sql("""
    SELECT * FROM brand_part_costs
    WHERE suppliers_offering > 1
    ORDER BY cost_spread_pct DESC
    LIMIT 10
"""))

In [0]:
q2_rank = spark.sql(f"""
    WITH ranked AS (
        SELECT *, RANK() OVER (PARTITION BY part_id ORDER BY supply_cost) AS cost_rank
        FROM {gold('part_supplier')}
        WHERE brand = '{BRAND}' AND suppliers_offering > 1
    )
    SELECT cost_rank,
           COUNT(*) AS offers,
           SUM(order_lines) AS order_lines,
           SUM(ordered_quantity) AS ordered_quantity,
           ROUND(SUM(ordered_quantity) / SUM(SUM(ordered_quantity)) OVER () * 100, 2) AS quantity_share_pct,
           SUM(spend) AS spend
    FROM ranked
    GROUP BY cost_rank
    ORDER BY cost_rank
""")
display(q2_rank)

display(spark.sql(f"""
    SELECT SUM(spend) AS actual_spend,
           SUM(ordered_quantity * min_supply_cost) AS spend_if_cheapest,
           SUM(spend) - SUM(ordered_quantity * min_supply_cost) AS overpay,
           ROUND((SUM(spend) - SUM(ordered_quantity * min_supply_cost)) / SUM(spend) * 100, 2) AS overpay_pct,
           ROUND(SUM(CASE WHEN is_cheapest THEN ordered_quantity ELSE 0 END) / SUM(ordered_quantity) * 100, 2)
               AS quantity_from_cheapest_pct
    FROM {gold('part_supplier')}
    WHERE brand = '{BRAND}' AND suppliers_offering > 1
"""))

In [0]:
spread_pd = spark.sql("SELECT cost_spread FROM brand_part_costs WHERE suppliers_offering > 1").toPandas()
rank_pd = q2_rank.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(spread_pd["cost_spread"].astype(float), bins=40)
axes[0].set_title(f"{BRAND}: supply cost spread per part")
axes[0].set_xlabel("Most expensive minus cheapest supply cost, per unit")
axes[0].set_ylabel("Parts")
axes[1].bar(rank_pd["cost_rank"].astype(str), rank_pd["quantity_share_pct"].astype(float))
axes[1].set_title(f"{BRAND}: ordered quantity by supplier cost rank")
axes[1].set_xlabel("Cost rank (1 = cheapest)")
axes[1].set_ylabel("Share of ordered quantity, %")
plt.tight_layout()
plt.show()

**Answer:** for a typical Brand#32 part the most expensive supplier costs **~4.8×** the cheapest (median spread +382%, average gap ~599 per unit). We are **not** ordering from the cheapest: quantity is split evenly across cost ranks (~25% each), so only **25%** goes to the cheapest. Spend was **15.25B**; with the cheapest supplier it would be **6.11B**, an overpay of **9.14B (60%)**.

## Q3. Parts ordered from a single supplier only

In [0]:
display(spark.sql(f"""
    SELECT suppliers_offering, suppliers_ordered, COUNT(DISTINCT part_id) AS parts
    FROM {gold('part_supplier')}
    GROUP BY suppliers_offering, suppliers_ordered
    ORDER BY suppliers_offering, suppliers_ordered
"""))

spark.sql(f"""
    SELECT ps.*, s.supplier_name, s.nation, s.region
    FROM {gold('part_supplier')} ps
    JOIN {gold('supplier')} s ON s.supplier_id = ps.supplier_id
    WHERE ps.suppliers_offering > 1
      AND ps.suppliers_ordered = 1
      AND ps.order_lines > 0
""").createOrReplaceTempView("single_sourced")

display(spark.sql(f"""
    SELECT COUNT(*) AS single_sourced_parts,
           COALESCE(SUM(spend), 0) AS single_sourced_spend,
           ROUND(COALESCE(SUM(spend), 0) / (SELECT SUM(spend) FROM {gold('part_supplier')}) * 100, 4) AS spend_share_pct,
           COUNT_IF(NOT is_cheapest) AS sole_supplier_not_cheapest
    FROM single_sourced
"""))

In [0]:
display(spark.sql(f"""
    WITH base AS (
        SELECT s.region, COUNT(*) AS ordered_offers
        FROM {gold('part_supplier')} ps
        JOIN {gold('supplier')} s ON s.supplier_id = ps.supplier_id
        WHERE ps.order_lines > 0
        GROUP BY s.region
    ),
    single AS (
        SELECT region, COUNT(*) AS single_sourced_parts
        FROM single_sourced
        GROUP BY region
    )
    SELECT b.region,
           COALESCE(s.single_sourced_parts, 0) AS single_sourced_parts,
           ROUND(COALESCE(s.single_sourced_parts, 0) / NULLIF(SUM(COALESCE(s.single_sourced_parts, 0)) OVER (), 0) * 100, 2)
               AS single_sourced_share_pct,
           ROUND(b.ordered_offers / SUM(b.ordered_offers) OVER () * 100, 2) AS ordered_offers_share_pct
    FROM base b
    LEFT JOIN single s ON s.region = b.region
    ORDER BY single_sourced_parts DESC
"""))

display(spark.sql("""
    SELECT manufacturer, COUNT(*) AS single_sourced_parts
    FROM single_sourced
    GROUP BY manufacturer
    ORDER BY single_sourced_parts DESC
"""))

display(spark.sql("""
    SELECT supplier_id, supplier_name, nation, region, COUNT(*) AS single_sourced_parts, SUM(spend) AS spend
    FROM single_sourced
    GROUP BY supplier_id, supplier_name, nation, region
    ORDER BY single_sourced_parts DESC, spend DESC
    LIMIT 10
"""))

In [0]:
dist_pd = spark.sql(f"""
    SELECT suppliers_ordered, COUNT(DISTINCT part_id) AS parts
    FROM {gold('part_supplier')}
    WHERE suppliers_offering > 1
    GROUP BY suppliers_ordered
    ORDER BY suppliers_ordered
""").toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dist_pd["suppliers_ordered"].astype(str), dist_pd["parts"])
ax.set_title("Parts with more than one supplier in partsupp: how many suppliers we ordered from")
ax.set_xlabel("Suppliers actually ordered from")
ax.set_ylabel("Parts")
plt.tight_layout()
plt.show()

**Answer:** **0 parts.** Every part with several suppliers in partsupp was ordered from at least 2 of them, so there is no single-sourcing risk in orders. The real exposure is in the data: 10 parts have only 1 valid supplier after quarantine.

## Q4. Suppliers with customer complaints

In [0]:
display(spark.sql(f"""
    SELECT supplier_id, supplier_name, nation, region, supplier_comment
    FROM {gold('supplier')}
    WHERE has_complaints
    ORDER BY supplier_id
"""))

q4 = spark.sql(f"""
    SELECT s.has_complaints,
           COUNT(*) AS suppliers,
           ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 4) AS suppliers_share_pct,
           SUM(COALESCE(m.spend, 0)) AS spend,
           ROUND(SUM(COALESCE(m.spend, 0)) / SUM(SUM(COALESCE(m.spend, 0))) OVER () * 100, 4) AS spend_share_pct
    FROM {gold('supplier')} s
    LEFT JOIN (
        SELECT supplier_id, SUM(spend) AS spend
        FROM {gold('supplier_spend_monthly')}
        GROUP BY supplier_id
    ) m ON m.supplier_id = s.supplier_id
    GROUP BY s.has_complaints
    ORDER BY s.has_complaints DESC
""")
display(q4)

In [0]:
def quarterly_supplier_spend(monthly_table: str) -> str:
    return f"""
        SELECT CAST(DATE_TRUNC('quarter', m.month) AS DATE) AS quarter_start,
               CONCAT(YEAR(m.month), '-Q', QUARTER(m.month)) AS quarter_label,
               m.supplier_id,
               m.region,
               SUM(m.quantity) AS quantity,
               SUM(m.spend) AS spend
        FROM {monthly_table} m
        JOIN complete_quarters c ON c.quarter_start = CAST(DATE_TRUNC('quarter', m.month) AS DATE)
        GROUP BY 1, 2, 3, 4
    """


complaints_trend = spark.sql(f"""
    WITH q AS ({quarterly_supplier_spend(gold('supplier_spend_monthly'))})
    SELECT q.quarter_start, q.quarter_label,
           SUM(CASE WHEN s.has_complaints THEN q.spend ELSE 0 END) / SUM(q.spend) * 100 AS complaints_spend_share_pct
    FROM q
    JOIN {gold('supplier')} s ON s.supplier_id = q.supplier_id
    GROUP BY q.quarter_start, q.quarter_label
    ORDER BY q.quarter_start
""").toPandas()

suppliers_share = float(q4.filter("has_complaints").first()["suppliers_share_pct"])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(complaints_trend["quarter_label"], complaints_trend["complaints_spend_share_pct"].astype(float),
        marker="o", label="Share of spend")
ax.axhline(suppliers_share, color="tab:gray", linestyle="--", label="Share of suppliers")
ax.set_title("Suppliers with customer complaints: share of spend per complete quarter")
ax.set_ylabel("%")
ax.tick_params(axis="x", rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

**Answer:** **26** suppliers have complaints. They get **202.5M of 382.3B (0.053%)** of spend, the same as their share of suppliers (0.052%), and it is stable over time. Complaints do not affect where we buy.

## Monitoring

Spend concentration (top-10 share) and average supply cost by region, per complete quarter.

In [0]:
concentration = spark.sql(f"""
    WITH q AS ({quarterly_supplier_spend(gold('supplier_spend_monthly'))}),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY quarter_start ORDER BY spend DESC) AS rn
        FROM q
    )
    SELECT quarter_start, quarter_label,
           CAST(SUM(CASE WHEN rn <= {TOP_N} THEN spend END) / SUM(spend) * 100 AS DOUBLE) AS top_share_pct
    FROM ranked
    GROUP BY quarter_start, quarter_label
    ORDER BY quarter_start
""").toPandas()

region_cost = spark.sql(f"""
    WITH q AS ({quarterly_supplier_spend(gold('supplier_spend_monthly'))})
    SELECT quarter_start, quarter_label, region,
           CAST(SUM(spend) / SUM(quantity) AS DOUBLE) AS avg_supply_cost
    FROM q
    GROUP BY quarter_start, quarter_label, region
    ORDER BY quarter_start, region
""").toPandas()

display(spark.createDataFrame(concentration.tail(4)))
display(spark.createDataFrame(region_cost.groupby("region")["avg_supply_cost"].agg(["min", "max"]).reset_index()))

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(concentration["quarter_label"], concentration["top_share_pct"].astype(float), marker="o")
axes[0].set_title(f"Spend concentration: share of top-{TOP_N} suppliers per complete quarter")
axes[0].set_ylabel("%")
for region, part in region_cost.groupby("region"):
    axes[1].plot(part["quarter_label"], part["avg_supply_cost"].astype(float), marker="o", label=region)
axes[1].set_title("Average supply cost per unit by supplier region")
axes[1].set_ylabel("Supply cost per unit")
axes[1].tick_params(axis="x", rotation=90)
axes[1].legend()
plt.tight_layout()
plt.show()

Top-10 share is stable at **~0.045%** and average supply cost is **~500** per unit in every region. No trend to act on.

## Alert: single supplier's share of spend

**Rule:** alert when one supplier gets more than **10× the equal share** (100% / number of suppliers) of spend in a complete quarter. The threshold scales with the number of suppliers.

In [0]:
SUPPLIER_COUNT = spark.table(gold("supplier")).count()
EQUAL_SHARE_PCT = 100 / SUPPLIER_COUNT
ALERT_MULTIPLIER = 10
ALERT_THRESHOLD_PCT = ALERT_MULTIPLIER * EQUAL_SHARE_PCT


def supplier_share_alert(monthly_table: str):
    return spark.sql(f"""
        WITH q AS ({quarterly_supplier_spend(monthly_table)})
        SELECT quarter_start, quarter_label, supplier_id, spend,
               CAST(spend / SUM(spend) OVER (PARTITION BY quarter_start) * 100 AS DOUBLE) AS share_pct,
               spend / SUM(spend) OVER (PARTITION BY quarter_start) * 100 > {ALERT_THRESHOLD_PCT} AS alert
        FROM q
    """)


real_shares = supplier_share_alert(gold("supplier_spend_monthly"))
real_shares.createOrReplaceTempView("real_supplier_shares")

display(spark.sql(f"""
    SELECT {SUPPLIER_COUNT} AS suppliers,
           ROUND({EQUAL_SHARE_PCT}, 6) AS equal_share_pct,
           ROUND({ALERT_THRESHOLD_PCT}, 6) AS alert_threshold_pct,
           ROUND(MAX(share_pct), 6) AS max_observed_share_pct,
           ROUND(MAX(share_pct) / {EQUAL_SHARE_PCT}, 2) AS max_observed_vs_equal
    FROM real_supplier_shares
"""))

real_alerts = real_shares.filter("alert")
display(real_alerts)
assert real_alerts.count() == 0, "Alert fires on real data"

Equal share is 0.002%, threshold 0.02%. The largest observed share is **0.0054% (2.7×)**, so the alert fires **0 times** on real data.

### Demo on corrupted data

Spend of the largest supplier in 1997-Q2 is multiplied by 10 (temp view, real tables unchanged).

In [0]:
DEMO_QUARTER = "1997-04-01"
DEMO_FACTOR = 10

demo_supplier = spark.sql(f"""
    SELECT supplier_id
    FROM {gold('supplier_spend_monthly')}
    WHERE CAST(DATE_TRUNC('quarter', month) AS DATE) = DATE '{DEMO_QUARTER}'
    GROUP BY supplier_id
    ORDER BY SUM(spend) DESC
    LIMIT 1
""").first()[0]

spark.sql(f"""
    SELECT month, supplier_id, region, order_lines, quantity,
           CASE WHEN supplier_id = {demo_supplier}
                 AND CAST(DATE_TRUNC('quarter', month) AS DATE) = DATE '{DEMO_QUARTER}'
                THEN spend * {DEMO_FACTOR}
                ELSE spend
           END AS spend
    FROM {gold('supplier_spend_monthly')}
""").createOrReplaceTempView("supplier_spend_monthly_demo")

demo_shares = supplier_share_alert("supplier_spend_monthly_demo")
demo_alerts = demo_shares.filter("alert")
display(demo_alerts)

fired = [(str(r["quarter_start"]), r["supplier_id"]) for r in demo_alerts.collect()]
assert fired == [(DEMO_QUARTER, demo_supplier)], f"Expected alert only for {demo_supplier} in {DEMO_QUARTER}, got {fired}"

In [0]:
max_share = lambda df: (df.groupBy("quarter_start", "quarter_label")
                          .agg({"share_pct": "max"})
                          .withColumnRenamed("max(share_pct)", "max_share_pct")
                          .orderBy("quarter_start")
                          .toPandas())

real_max = max_share(real_shares)
demo_max = max_share(demo_shares)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(real_max["quarter_label"], real_max["max_share_pct"].astype(float), marker="o", label="Real data")
ax.plot(demo_max["quarter_label"], demo_max["max_share_pct"].astype(float), marker="o", linestyle="--",
        label=f"Demo: supplier {demo_supplier} ×{DEMO_FACTOR}")
ax.axhline(ALERT_THRESHOLD_PCT, color="tab:red", linestyle=":", label=f"Threshold {ALERT_THRESHOLD_PCT:.4f}%")
ax.set_title("Largest single-supplier share of spend per complete quarter")
ax.set_ylabel("%")
ax.tick_params(axis="x", rotation=90)
ax.legend()
plt.tight_layout()
plt.show()

**Result:** the alert fires **only** for supplier 4713 in 1997-Q2 (share 0.048%, above the 0.02% threshold).